# WSJ 2027 - Gruppindelning Direktresa

Assign confirmed direktresa deltagare into groups of exactly 36.

Direktresa participants travel independently to/from WSJ in Poland.
Same grouping algorithm as rundresa but applied to the direktresa subset.

## Rules
1. **Exactly 36 per group** (remainder group allowed)
2. **Geographic proximity** — Hilbert curve sort
3. **Friend wish** — at least ONE friend in same group (soft goal)
4. **Max 8 from same kår** per group (hard constraint)
5. **Diversity** — age (14-17) and sex balance

In [1]:
import sys
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u

# Fetch and parse all participants
raw_data = u.fetch_participants()
df, skipped = u.build_participant_dataframe(raw_data)

Fetched 2478 participants
Confirmed: 2413, Unconfirmed: 3, Cancelled: 62
Total confirmed participants: 2413
Skipped: 3 unconfirmed, 0 wrong age/no DOB

By category:
category
deltagare    1940
ist           451
cmt            22

By travel type:
travel
rundresa      1515
direktresa     572
egen_resa      304
other           22

Friend wishes:
  With friend 1 member no: 1358
  With friend 2 member no: 805
  With friend 1 name (text): 118
  With friend 2 name (text): 80


In [2]:
GROUP_SIZE = 36

# Filter to direktresa deltagare only
df_direkt = df[df['travel'] == 'direktresa'].copy().reset_index(drop=True)
print(f"=== Direktresa deltagare ===")
print(f"Participants: {len(df_direkt)}")

# Assign coordinates and Hilbert sort
u.assign_coordinates(df_direkt)
df_sorted = u.add_hilbert_index(df_direkt)

n_full = len(df_sorted) // GROUP_SIZE
remainder = len(df_sorted) % GROUP_SIZE
total_groups = n_full + (1 if remainder > 0 else 0)
print(f"\nGroups: {n_full} x {GROUP_SIZE} + 1 x {remainder} = {total_groups} total")
print(f"\nBy region:")
print(df_sorted['region'].value_counts().to_string())
print(f"\nBy age:")
print(df_sorted['age'].value_counts().sort_index().to_string())
print(f"\nBy sex:")
print(df_sorted['sex'].map(u.SEX_MAP).value_counts().to_string())

=== Direktresa deltagare ===
Participants: 572
With coordinates: 569
Without coordinates (Sweden centroid): 3

Groups: 15 x 36 + 1 x 32 = 16 total

By region:
region
Region Stockholm    252
Region Södra         91
Region Norr/Mitt     83
Region Västra        67
Region Östra         37

By age:
age
14    170
15    174
16    139
17     89

By sex:
sex
Man       301
Kvinna    265
Annat       6


In [3]:
# Resolve text-only friend wishes via fuzzy name matching
u.resolve_friend_wishes(df_sorted, df)
friend_wishes = u.build_friend_graph(df_sorted)

=== Text Friend Name Matching ===
Text-only wishes: 45
Matched & applied: 37
Generic wishes (not a person): 0
Unresolved (friend not in project): 8

Matched:
  Alida Vakk -> Freja Jernberg (Örnsbergs Scoutkår) [rundresa] via fuzzy(0.93)+kar
  Alma Hagström -> Vera Turesson (Equmenia Scout) [direktresa] via fuzzy(0.96)
  Alma Hagström -> Vilda Englund (Equmenia Scout) [direktresa] via exact
  Alma Oliver Elgh -> Elsa Mattsson (Vendelsö Scoutkår) [direktresa] via exact
  Alma Oliver Elgh -> Elsa Mattsson (Vendelsö Scoutkår) [direktresa] via fuzzy(0.92)+kar
  Armilde Westerblom -> Loke Jageteg (Falkenbergs Scoutkår) [rundresa] via exact
  Armilde Westerblom -> Vera Hedgren (Bohus Scoutkår) [rundresa] via exact
  Arvid Bergsten -> Samuel Nordström (Tollereds Scoutkår) [direktresa] via exact
  Arvid Bergsten -> Eskil Norrfjärd (Equmenia Scout) [rundresa] via exact
  Astrid Dodd -> Gustav Glimtoft (Älvsjö Scoutkår) [direktresa] via exact
  Elin Hjertstedt -> Hilma Wixner (Tegs Scoutkår) [dir

In [4]:
# Run the full group assignment algorithm
df_sorted = u.assign_groups(df_sorted, GROUP_SIZE, friend_wishes)
total_groups = df_sorted['group'].nunique()

Participants: 572
Groups: 15 x 36 + 1 x 32 = 16 total

=== Phase 1: Geographic sort + cut ===
  Friend satisfaction: 312/338
  Kar violations: 15
  Avg geo spread: 1.7010

=== Phase 2: Fix friend wishes ===
  Swaps: 15
  Friend satisfaction: 336/338
  Kar violations: 14
  Avg geo spread: 1.7884

=== Phase 3: Fix kar violations (geo-aware) ===
  Swaps: 14
  Kar violations: 0
  Friend satisfaction: 319/338
  Avg geo spread: 1.7810

=== Phase 2b: Re-fix friends after kar fix ===
  Swaps: 9
  Friend satisfaction: 336/338
  Kar violations: 0
  Avg geo spread: 1.7821

=== Phase 4: Diversity optimization (geo-penalized) ===
  Swaps: 329
  Diversity score: 46.09 -> 46.93
  Avg geo spread: 1.7821 -> 2.1010

=== FINAL RESULTS ===
Groups: 15 x 36 + 1 x 32
Friend satisfaction: 337/338 (100%)
Kar violations: 0
Total swaps: 367
Diversity: 46.93
Avg geo spread: 2.1010


In [5]:
# Per-group quality metrics
group_of = df_sorted['group'].values
u.print_group_metrics(df_sorted, group_of, total_groups)

Grupp Storlek  Van% MaxKar     M/K/A  14  15  16  17  AvstandKm Karer
--------------------------------------------------------------------------------
    1      36 14/14      4   15/20/1  12  11   9   4        61    20
        Orter: Höörs (4), Eslövs (4), Dalby (3), Södra Sandby (3), Finn (3), Trelleborgs (2), Veberöd (2), Kirsebergs (2), Åhus (2), Hörby (1), Karlsro (1), Tornugglan (1), Djupadals (1), Bunkeflo (1), Bjärred (1), Kävlinge (1), Viskafors (1), Björsäters (1), Equmenia Rimforsa (1), Landeryd (1)
    2      36 13/13      4   19/16/1   8  13  10   5        65    24
        Orter: Djupadals (4), Slottsstadens (3), Tornfalkens (3), Bjärred (3), Trelleborgs (2), Tollarps (2), Östra Ljungby (2), Vellinge (1), Dalby (1), Genarps (1), Höörs (1), Svedala (1), Equmenia Toarp (1), Husie-Hohögs (1), Bunkeflo (1), Scouterna Lödde (1), Kävlinge (1), Landskrona KM (1), Spejaren - Klippan (1), Varbergs (1), Redbergslids (1), Trollhättans (1), Kärna (1), Equmenia Linköping (1)
    3     

In [6]:
# Export CSV + JSON
OUTPUT_DIR = '/config/notebooks/wsj27/output'
csv_path, json_path = u.export_results(df_sorted, group_of, total_groups, OUTPUT_DIR, prefix='wsj27_direktresa')

Saved 572 participants to /config/notebooks/wsj27/output/wsj27_direktresa_grupper.csv
Saved group summary to /config/notebooks/wsj27/output/wsj27_direktresa_grupper.json

CSV preview (first 10 rows):
 group               name member_no  age    sex                 kar                          district       region friend_1 friend_2       lat       lng
     1       Nils Calling   3342484   15    Man    Bjärred scoutkår        Södra Skånes Scoutdistrikt Region Södra                   55.722186 13.021644
     1    Jasmine Båtwijk   3329872   14 Kvinna Björsäters Scoutkår Nordöstra Götalands Scoutdistrikt Region Östra                   58.655167 13.698961
     1      Edit Fryklund   3351167   15 Kvinna   Bunkeflo Scoutkår        Södra Skånes Scoutdistrikt Region Södra                   55.557482 12.963234
     1 Lindsey Bäverkvist   3356950   16 Kvinna      Dalby Scoutkår        Södra Skånes Scoutdistrikt Region Södra  3451529          55.664664 13.346159
     1          Liv Kropp   3421219

In [7]:
# Generate interactive map
map_path = f'{OUTPUT_DIR}/wsj_direktresa_karta.html'
u.generate_group_map_html(df_sorted, total_groups, map_path, title='WSJ 2027 Direktresa',
                          friend_wishes=friend_wishes, show_group_arcs=True)
print(f"\nAll outputs:")
print(f"  CSV:  {csv_path}")
print(f"  JSON: {json_path}")
print(f"  Map:  {map_path}")

Friend arcs: 295 (279 satisfied, 16 unsatisfied)
Group arcs: 9946 connections across 16 groups
Saved group map: /config/notebooks/wsj27/output/wsj_direktresa_karta.html

All outputs:
  CSV:  /config/notebooks/wsj27/output/wsj27_direktresa_grupper.csv
  JSON: /config/notebooks/wsj27/output/wsj27_direktresa_grupper.json
  Map:  /config/notebooks/wsj27/output/wsj_direktresa_karta.html
